In [33]:
import copy
import os
import argparse
from erasure.utils.logger import GLogger
import torch
import numpy as np
import random
from erasure.utils.config.local_ctx import Local
from erasure.utils.config.global_ctx import Global, bcolors 
from erasure.core.factory_base import ConfigurableFactory
from erasure.data.datasets.DatasetManager import DatasetManager

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
config_file = os.path.join("configs","LinkAttack", "Citeseer_edge.jsonc")

In [35]:
global_ctx = Global(config_file)
global_ctx.factory = ConfigurableFactory(global_ctx)

#Create Dataset
data_manager = global_ctx.factory.get_object( Local( global_ctx.config.data ))
global_ctx.dataset = data_manager

@,-724899871 | INFO | 3737319 - Creating Global Context for: configs/LinkAttack/Citeseer_edge.jsonc
���ax,-724899868 | INFO | 3737319 - Setting seeds to: 0
!,-724899859 | INFO | 3737319 - REMOVAL TYPE SET AS edge
,-724899858 | INFO | 3737319 - Caching System: False.
8�e�y,-724899855 | INFO | 3737319 - Instantiating: torch_geometric.datasets.Planetoid


0� x,-724899670 | INFO | 3737319 - Created Configurable: erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
p�s,-724899669 | INFO | 3737319 - {'class': 'erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.Planetoid', 'parameters': {'root': 'resources/data', 'name': 'Citeseer'}, 'preprocess': []}}}
8�e�y,-724899608 | INFO | 3737319 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
�0W�y,-724899556 | INFO | 3737319 - ['all', 'all_shuffled', '-']
8�e�y,-724899555 | INFO | 3737319 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
��J,-724899547 | INFO | 3737319 - ['all', 'all_shuffled', '-', 'train_0', 'test']
8�e�y,-724899529 | INFO | 3737319 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
�I,-724899515 | INFO | 3737319 - ['all', 'all_shuffled', '-', 'train_0', 'test', 'validati

In [36]:
#Create Predictor
current = Local(global_ctx.config.predictor)
current.dataset = data_manager
predictor = global_ctx.factory.get_object(current)
global_ctx.predictor = predictor
global_ctx.logger.info('Global Predictor: ' + str(predictor))

8�e�y,-724899428 | INFO | 3737319 - Instantiating: erasure.model.graphs.GCN.GCN


8�e�y,-724899420 | INFO | 3737319 - Instantiating: torch.optim.Adam
8�e�y,-724899419 | INFO | 3737319 - Instantiating: torch.nn.CrossEntropyLoss
graph has edges  torch.Size([2, 9104])
��t,-724899334 | INFO | 3737319 - epoch = 0 ---> loss = 1.7978	 accuracy = 0.1720
graph has edges  torch.Size([2, 9104])
��t,-724899278 | INFO | 3737319 - epoch = 1 ---> loss = 1.7470	 accuracy = 0.3599
graph has edges  torch.Size([2, 9104])
��t,-724899223 | INFO | 3737319 - epoch = 2 ---> loss = 1.6973	 accuracy = 0.5436
graph has edges  torch.Size([2, 9104])
��t,-724899168 | INFO | 3737319 - epoch = 3 ---> loss = 1.6504	 accuracy = 0.6200
graph has edges  torch.Size([2, 9104])
��t,-724899111 | INFO | 3737319 - epoch = 4 ---> loss = 1.5992	 accuracy = 0.6785
graph has edges  torch.Size([2, 9104])
��t,-724899056 | INFO | 3737319 - epoch = 5 ---> loss = 1.5530	 accuracy = 0.7027
graph has edges  torch.Size([2, 9104])
��t,-724899002 | INFO | 3737319 - epoch = 6 ---> loss = 1.5004	 accuracy = 0.7219
graph 

In [37]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [38]:
data_manager.partitions['all'][0][1]

tensor([3, 1, 5,  ..., 3, 1, 5])

In [39]:
import torch
print(torch.version.cuda)

12.6


In [40]:
print(predictor.optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.0005000000000000002
    maximize: False
    weight_decay: 0
)


In [41]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.7.0+cu126
12.6


In [42]:
#Create unlearners 
unlearners = []
unlearners_cfg = global_ctx.config.unlearners
for un in unlearners_cfg:
    current = Local(un)
    current.dataset = data_manager
    current.predictor = copy.deepcopy(predictor)
    unlearners.append( global_ctx.factory.get_object(current) )

8�e�y,-724896091 | INFO | 3737319 - Instantiating: torch.optim.SGD
0� x,-724896089 | INFO | 3737319 - Created Configurable: erasure.unlearners.graph_unlearners.BadTeaching.BadTeaching


In [43]:
data_manager.partitions['all'][0][0]

Data(x=[3327, 3703], edge_index=[2, 9104])

In [44]:
data_manager.partitions['all'][0][0].edge_index

tensor([[   0,    1,    1,  ..., 3324, 3325, 3326],
        [ 628,  158,  486,  ..., 2820, 1643,   33]])

In [45]:
data_manager.partitions['all'].revise_graph_edges([(0,1),(0,2)])[0][0]

Data(x=[3327, 3703], edge_index=[2, 0])

In [46]:
print(len( data_manager.partitions['forget']) )

1820


In [48]:
#Evaluator
current = Local(global_ctx.config.evaluator)
current.unlearners = unlearners
evaluator = global_ctx.factory.get_object(current)

# Evaluations
for unlearner in unlearners:
    global_ctx.logger.info(f'''{bcolors.OKGREEN}####\t\t Evaluating Unlearner {unlearner.__class__.__name__} \t\t####{bcolors.ENDC}''')
    evaluator.evaluate(unlearner,predictor)

0� x,-724457324 | INFO | 3737319 - Created Configurable: erasure.evaluations.running.RunTime
�#D,-724457321 | INFO | 3737319 - Function: <function accuracy_score at 0x7f780079b3a0>
0� x,-724457321 | INFO | 3737319 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
0@P,-724457320 | INFO | 3737319 - Function: <function accuracy_score at 0x7f780079b3a0>
0� x,-724457319 | INFO | 3737319 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
��,-724457318 | INFO | 3737319 - Function: <function accuracy_score at 0x7f780079b3a0>
0� x,-724457318 | INFO | 3737319 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
�^,-724457317 | INFO | 3737319 - Function: <function accuracy_score at 0x7f780079b3a0>
0� x,-724457316 | INFO | 3737319 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
0� x,-724457315 | INFO | 3737319 - Created Configurable: erasure.evaluations.measures.SaveValues
0� x,-724457315 | INFO | 373

/NFSHOME/adangelo/LinkInference/erasure/unlearners/graph_unlearners/BadTeaching.py:106: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  all_nodes = torch.arange(num_nodes)


,-724436444 | INFO | 3737319 - Epoch 0 Unlearning Loss 0.9887118339538574
�ݜx,-724436410 | INFO | 3737319 - sklearn.metrics.accuracy_score on partition: "test", target model: unlearned, unlearned graph: True: 0.7537537537537538 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f78006e41f0>
�ݜx,-724436376 | INFO | 3737319 - sklearn.metrics.accuracy_score on partition: "test", target model: original, unlearned graph: True: 0.7507507507507507 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f780031ff10>
�ݜx,-724436345 | INFO | 3737319 - sklearn.metrics.accuracy_score on partition: "test", target model: unlearned, unlearned graph: False: 0.7567567567567568 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f78006e41f0>
�ݜx,-724436314 | INFO | 3737319 - sklearn.metrics.accuracy_score on partition: "test", target model: original, unlearned graph: False: 0.7582582582582582 of <erasure.model.TorchGraphModel.TorchGraphModel object at 0x7f780031ff10